# Huấn luyện và Đánh giá Baseline NAFNet (Multi-Task Restoration)

In [1]:
# ==============================================================================
# CELL 1: SETUP MÔI TRƯỜNG VÀ IMPORT THƯ VIỆN
# ==============================================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import math
import sys
import time
import random
import gc
import json
import shutil
import hashlib
import urllib.request
import glob
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Bật hỗ trợ Mixed Precision (FP16) chuẩn PyTorch 2.x mới nhất
from torch.amp import autocast, GradScaler

# Tạo thư mục weights lưu checkpoint
os.makedirs("weights", exist_ok=True)

candidate_paths = [
    '.', '..', 
    '/kaggle/working', 
    '/kaggle/working/Image-processing-project-integrating-MoE-and-Q-Former'
]

for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'models')) and os.path.exists(os.path.join(p, 'utils')):
        abs_p = os.path.abspath(p)
        if abs_p not in sys.path:
            sys.path.insert(0, abs_p)
        break

try:
    from models.forge import NAFNetBaseline
    from utils.metrics import CharbonnierLoss, FFTLoss, DeblurCompositeLoss, calc_psnr, calc_ssim, count_parameters
    from utils.dataset import MultiTaskRestorationDataset, get_multitask_balanced_splits
except ImportError as e:
    print(f"Cảnh báo Import: {e}. Vui lòng kiểm tra lại cấu trúc thư mục models/ và utils/.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

# Download Pretrained Baseline NAFNet
pretrained_url = "https://huggingface.co/Jacid23/third-eye-models/resolve/5c2e322323888d8d26852f06715ccc6eb61ea795/NAFNet-GoPro-width64.pth"
nafnet_ckpt_path = os.path.join("weights", "NAFNet-GoPro-width64.pth")

if not os.path.exists(nafnet_ckpt_path):
    print("--> Đang tải pretrained NAFNet từ Hugging Face...")
    urllib.request.urlretrieve(pretrained_url, nafnet_ckpt_path)
    print("--> Tải xuống checkpoint thành công!")
else:
    print(f"--> Đã tìm thấy checkpoint tại: {nafnet_ckpt_path}")

nafnet_state_dict = torch.load(nafnet_ckpt_path, map_location=device)
if 'params' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['params']
elif 'model_state_dict' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['model_state_dict']
print("--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!")

Thiết bị đang sử dụng: cuda
--> Đã tìm thấy checkpoint tại: weights\NAFNet-GoPro-width64.pth
--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!


In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO DATALOADER MULTI-TASK
# ==============================================================================
def auto_detect_multitask_dataset():
    candidate_names = ['dataset_multitask_unified', 'multitask_dataset']
    prefixes = ['.', '..', '/kaggle/input/*', '/kaggle/input']
    
    for prefix in prefixes:
        for path in glob.glob(prefix):
            for c_name in candidate_names:
                full_p = os.path.join(path, c_name)
                if os.path.exists(full_p):
                    return os.path.abspath(full_p)
    return 'dataset_multitask_unified'

multitask_dir = auto_detect_multitask_dataset()
print(f"--> Đã tìm thấy đường dẫn Multitask Dataset: {multitask_dir}")

MAX_TRAIN_SAMPLES = 3000
MAX_TEST_SAMPLES = 300

train_samples, val_samples, test_samples = get_multitask_balanced_splits(
    root_dir=multitask_dir,
    val_ratio=0.15,
    seed=42,
    max_train_samples_per_task=MAX_TRAIN_SAMPLES,
    max_test_samples_per_task=MAX_TEST_SAMPLES
)

train_sub_ds = MultiTaskRestorationDataset(samples=train_samples, crop_size=192, is_train=True)
val_sub_ds = MultiTaskRestorationDataset(samples=val_samples, crop_size=256, is_train=False)
test_sub_ds = MultiTaskRestorationDataset(samples=test_samples, crop_size=256, is_train=False)

num_workers_cfg = 0 if os.name == 'nt' else 2
train_loader = DataLoader(train_sub_ds, batch_size=4, shuffle=True, num_workers=num_workers_cfg, pin_memory=True)
val_loader = DataLoader(val_sub_ds, batch_size=1, shuffle=False, num_workers=num_workers_cfg, pin_memory=True)
test_loader = DataLoader(test_sub_ds, batch_size=1, shuffle=False, num_workers=num_workers_cfg, pin_memory=True)

print(f"--> Khởi tạo DataLoader thành công: Train={len(train_sub_ds)} | Val={len(val_sub_ds)} | Test={len(test_sub_ds)}")

--> Đã tìm thấy đường dẫn Multitask Dataset: e:\DuAn\Image-processing-project-integrating-MoE-and-Q-Former\dataset_multitask_unified
--> [FAST LOAD] Đã nạp thành công split từ file Cache: multitask_splits_cache.json
--> Khởi tạo thành công DataLoader: Train=4687 | Val=892 | Test=942


In [ ]:
# ==============================================================================
# CELL 3: KIỂM TRA DATA LEAKAGE
# ==============================================================================
def get_file_md5(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

def audit_split_data_leakage(train_samples, val_samples, test_samples):
    print("=" * 80)
    print("🔍 AUDIT CHỐNG DATA LEAKAGE CÁC TẬP TRAIN / VAL / TEST (MD5 Target Hash)")
    print("=" * 80)

    all_tasks = sorted(list(set([s[3] for s in train_samples + val_samples + test_samples])))
    total_leaks = 0

    for task_name in all_tasks:
        tr_task = [s for s in train_samples if s[3] == task_name]
        val_task = [s for s in val_samples if s[3] == task_name]
        te_task = [s for s in test_samples if s[3] == task_name]

        tr_md5_map = {get_file_md5(s[1]): s[1] for s in tr_task}
        val_md5_map = {get_file_md5(s[1]): s[1] for s in val_task}
        te_md5_map = {get_file_md5(s[1]): s[1] for s in te_task}

        tr_val_overlap = set(tr_md5_map.keys()).intersection(set(val_md5_map.keys()))
        tr_te_overlap = set(tr_md5_map.keys()).intersection(set(te_md5_map.keys()))
        val_te_overlap = set(val_md5_map.keys()).intersection(set(te_md5_map.keys()))

        task_leaks = len(tr_val_overlap) + len(tr_te_overlap) + len(val_te_overlap)
        total_leaks += task_leaks

        print(f"\n📌 TASK [{task_name.upper()}]:")
        print(f"  • Mẫu Dữ Liệu : Train = {len(tr_task):4d} | Val = {len(val_task):3d} | Test = {len(te_task):3d}")
        print(f"  • Target Clean: Train = {len(tr_md5_map):4d} | Val = {len(val_md5_map):3d} | Test = {len(te_md5_map):3d}")
        if tr_val_overlap:
            print(f"  🚨 LEAK TRAIN - VAL : Phát hiện {len(tr_val_overlap)} ảnh Clean trùng nội dung!")
        else:
            print("  ✅ Train vs Val     : 0% Leakage (Hoàn toàn độc lập)")

        if tr_te_overlap:
            print(f"  🚨 LEAK TRAIN - TEST: Phát hiện {len(tr_te_overlap)} ảnh Clean trùng nội dung!")
        else:
            print("  ✅ Train vs Test    : 0% Leakage (Hoàn toàn độc lập)")

        if val_te_overlap:
            print(f"  🚨 LEAK VAL - TEST  : Phát hiện {len(val_te_overlap)} ảnh Clean trùng nội dung!")
        else:
            print("  ✅ Val vs Test      : 0% Leakage (Hoàn toàn độc lập)")
    print("=" * 80)
    if total_leaks == 0:
        print("🎉 TẤT CẢ TÁC VỤ ĐẠT CHUẨN LEAKPROOF 100%!")
    print("=" * 80)

audit_split_data_leakage(train_samples, val_samples, test_samples)

🔍 BÁO CÁO AUDIT CHỐNG DATA LEAKAGE CÁC TẬP TRAIN / VAL / TEST (MD5 Target Hash)

📌 TASK [DEBLUR]:
  • Mẫu Dữ Liệu : Train = 1010 | Val = 180 | Test = 200
  • Target Clean: Train = 1010 | Val = 180 | Test = 200 (MD5 độc lập)
  ✅ Train vs Val     : 0% Leakage (Hoàn toàn độc lập)
  ✅ Train vs Test    : 0% Leakage (Hoàn toàn độc lập)
  ✅ Val vs Test      : 0% Leakage (Hoàn toàn độc lập)

📌 TASK [DEHAZE]:
  • Mẫu Dữ Liệu : Train =  975 | Val = 210 | Test = 200
  • Target Clean: Train =   28 | Val =   6 | Test =   6 (MD5 độc lập)
  ✅ Train vs Val     : 0% Leakage (Hoàn toàn độc lập)
  ✅ Train vs Test    : 0% Leakage (Hoàn toàn độc lập)
  ✅ Val vs Test      : 0% Leakage (Hoàn toàn độc lập)

📌 TASK [DENOISE]:
  • Mẫu Dữ Liệu : Train =   83 | Val =  34 | Test =  42
  • Target Clean: Train =   83 | Val =  34 | Test =  42 (MD5 độc lập)
  ✅ Train vs Val     : 0% Leakage (Hoàn toàn độc lập)
  ✅ Train vs Test    : 0% Leakage (Hoàn toàn độc lập)
  ✅ Val vs Test      : 0% Leakage (Hoàn toàn độc lập)



In [ ]:
# ==============================================================================
# CELL 4: HÀM ĐÁNH GIÁ MÔ HÌNH
# ==============================================================================
def evaluate_multitask_model(model, loader, desc="Validating", max_size=1024):
    model.eval()
    task_psnr = defaultdict(list)
    task_ssim = defaultdict(list)
    multiple = 16 
    pbar_eval = tqdm(loader, desc=f"[{desc}]", leave=False)
    
    with torch.no_grad():
        for inputs, targets, task_ids, task_names in pbar_eval:
            inputs, targets = inputs.to(device), targets.to(device)

            _, _, h_orig, w_orig = inputs.shape
            if h_orig > max_size or w_orig > max_size:
                h_crop = min(h_orig, max_size)
                w_crop = min(w_orig, max_size)
                top = (h_orig - h_crop) // 2
                left = (w_orig - w_crop) // 2
                inputs = inputs[:, :, top:top+h_crop, left:left+w_crop]
                targets = targets[:, :, top:top+h_crop, left:left+w_crop]

            h, w = inputs.shape[2], inputs.shape[3]
            H = ((h + multiple - 1) // multiple) * multiple
            W = ((w + multiple - 1) // multiple) * multiple
            pad_h, pad_w = H - h, W - w
            inputs_padded = F.pad(inputs, (0, pad_w, 0, pad_h), mode='reflect')

            out = model(inputs_padded.float())
            preds = out[0] if isinstance(out, tuple) else out
            preds = preds[:, :, :h, :w]
            preds = torch.clamp(preds, 0.0, 1.0)
            
            p_val = calc_psnr(preds, targets)
            s_val = calc_ssim(preds, targets)
            
            p_item = p_val.item() if torch.is_tensor(p_val) else p_val
            s_item = s_val.item() if torch.is_tensor(s_val) else s_val

            for t_name in task_names:
                if not math.isnan(p_item):
                    task_psnr[t_name].append(p_item)
                if not math.isnan(s_item):
                    task_ssim[t_name].append(s_item)

    all_psnr = [val for sublist in task_psnr.values() for val in sublist if not math.isnan(val)]
    all_ssim = [val for sublist in task_ssim.values() for val in sublist if not math.isnan(val)]
    
    avg_psnr = sum(all_psnr) / len(all_psnr) if all_psnr else 0.0
    avg_ssim = sum(all_ssim) / len(all_ssim) if all_ssim else 0.0

    task_summary = {}
    for t_name in task_psnr:
        p_list = [v for v in task_psnr[t_name] if not math.isnan(v)]
        s_list = [v for v in task_ssim[t_name] if not math.isnan(v)]
        t_p = sum(p_list) / len(p_list) if p_list else 0.0
        t_s = sum(s_list) / len(s_list) if s_list else 0.0
        task_summary[t_name] = {'psnr': t_p, 'ssim': t_s}

    return avg_psnr, avg_ssim, task_summary

In [ ]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN BASELINE NAFNET (ĐÃ SỬA LỖI TUPLE UNPACKING & CHUẨN HÓA AMP)
# ==============================================================================
# Khởi tạo mô hình Baseline NAFNet
baseline_model = NAFNetBaseline(
    img_channel=3, 
    width=64, 
    enc_blk_nums=[1, 1, 1, 28], 
    middle_blk_num=1, 
    dec_blk_nums=[1, 1, 1, 1]
).to(device)

# Load pretrained weights từ GoPro
baseline_model.load_state_dict(nafnet_state_dict, strict=True)

# Cấu hình tham số huấn luyện
BASELINE_EPOCHS = 25
BASELINE_LR = 1e-4

optimizer_base = optim.AdamW(baseline_model.parameters(), lr=BASELINE_LR, weight_decay=1e-4)
scheduler_base = optim.lr_scheduler.CosineAnnealingLR(optimizer_base, T_max=BASELINE_EPOCHS, eta_min=1e-6)
criterion_base = DeblurCompositeLoss(eps=1e-3, fft_weight=0.05, vgg_weight=0.0)

# Khởi tạo GradScaler theo chuẩn PyTorch mới (tránh Deprecation Warning)
scaler = GradScaler('cuda')

best_val_psnr_base = 0.0
best_baseline_ckpt = os.path.join("weights", "best_baseline_nafnet.pth")

print("\n===================================================")
print(f" BẮT ĐẦU HUẤN LUYỆN BASELINE NAFNET ({BASELINE_EPOCHS} EPOCHS - AMP FP16)")
print("===================================================")

for epoch in range(1, BASELINE_EPOCHS + 1):
    baseline_model.train()
    running_loss = 0.0
    valid_batches = 0
    
    pbar = tqdm(train_loader, desc=f"[Train Baseline] Epoch {epoch:02d}/{BASELINE_EPOCHS:02d}")
    
    for inputs, targets, task_ids, task_names in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer_base.zero_grad(set_to_none=True)
        
        # Autocast chuẩn PyTorch 2.x
        with autocast('cuda'):
            preds = baseline_model(inputs.float())
            # GIẢI QUYẾT LỖI: Unpack tuple nếu mô hình trả về (output, ...)
            preds = preds[0] if isinstance(preds, tuple) else preds
            loss = criterion_base(preds, targets.float())
        
        if torch.isnan(loss) or torch.isinf(loss):
            continue
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer_base)
        torch.nn.utils.clip_grad_norm_(baseline_model.parameters(), max_norm=1.0)
        scaler.step(optimizer_base)
        scaler.update()
        
        running_loss += loss.item()
        valid_batches += 1
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{optimizer_base.param_groups[0]['lr']:.2e}"})
        
    epoch_loss = running_loss / max(1, valid_batches)
    scheduler_base.step()
    
    # Đánh giá sau mỗi epoch trên tập Validation
    val_psnr, val_ssim, _ = evaluate_multitask_model(baseline_model, val_loader, desc="Validation Baseline")
    print(f"--> [Epoch {epoch:02d}/{BASELINE_EPOCHS:02d}] Loss: {epoch_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | SSIM: {val_ssim:.4f}")
    
    # Lưu Checkpoint tốt nhất
    if val_psnr > best_val_psnr_base:
        best_val_psnr_base = val_psnr
        torch.save(baseline_model.state_dict(), best_baseline_ckpt)
        print(f"    [+] Đã lưu checkpoint Baseline tốt nhất vào: {best_baseline_ckpt}")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# CELL 6: ĐÁNH GIÁ MÔ HÌNH BASELINE TRÊN TẬP TEST VÀ LƯU FILE KẾT QUẢ
# ==============================================================================
if os.path.exists(best_baseline_ckpt):
    baseline_model.load_state_dict(torch.load(best_baseline_ckpt, map_location=device))
    print(f"--> Đã nạp thành công weights tốt nhất từ: {best_baseline_ckpt}")

print("\n===================================================")
print(" ĐÁNH GIÁ NAFNet BASELINE TRÊN TẬP TEST MULTITASK")
print("===================================================")
baseline_psnr, baseline_ssim, baseline_summary = evaluate_multitask_model(baseline_model, test_loader, desc="Baseline Test")

print(f"\n KẾT QUẢ NAFNET BASELINE (TRUNG BÌNH CÁC TASK):")
print(f"   + PSNR Trung bình : {baseline_psnr:.2f} dB")
print(f"   + SSIM Trung bình : {baseline_ssim:.4f}")
print("\n CHI TIẾT THEO TỪNG TASK:")
for t_name, metrics in sorted(baseline_summary.items()):
    print(f"   - Task [{t_name:8s}]: PSNR = {metrics['psnr']:.2f} dB | SSIM = {metrics['ssim']:.4f}")

# Lưu kết quả Baseline ra file JSON
with open("weights/baseline_results.json", "w") as f:
    json.dump({
        "avg_psnr": baseline_psnr,
        "avg_ssim": baseline_ssim,
        "tasks": baseline_summary
    }, f, indent=4)

print("\n--> Đã lưu kết quả Baseline vào file weights/baseline_results.json")